# Normalisation in Quant Finance for Beginners

This notebook explains the idea of **normalisation** in quant finance in a beginner-friendly way. The goal is to understand what normalisation is, why quants use it, and when different normalisation methods are appropriate.

## What you will learn

- what normalisation means in simple terms
- why raw financial data is often hard to compare directly
- the difference between scaling, standardisation, and return normalisation
- when to use min-max scaling, z-scores, and rebasing
- how normalisation helps with risk analysis, signals, and machine learning

## Beginner intuition

Normalisation is about putting data onto a more comparable scale. In finance, different assets can have different price levels, different volatility, and different units. If you compare them without adjustment, you can easily misread the data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 11

## 1. Why Normalisation Matters

Raw financial data can be misleading when assets live on different scales. A stock trading at $20 and another trading at $2,000 are not directly comparable by price level alone.

### Beginner explanation

Suppose two assets both rise by 5%, but one starts at 10 and the other starts at 1,000. Their **price changes** look very different in absolute terms, but their **economic move** is the same in percentage terms. Normalisation helps you compare them more fairly.

In quant finance, this matters for:

- comparing different assets
- building signals from multiple features
- clustering assets
- machine learning models that are sensitive to feature scale

In [ ]:
n_days = 252
dates = pd.bdate_range("2024-01-01", periods=n_days)

price_df = pd.DataFrame({
    "LOW_PRICE_STOCK": 20 * np.exp(np.cumsum(np.random.normal(0.0005, 0.018, n_days))),
    "HIGH_PRICE_STOCK": 1200 * np.exp(np.cumsum(np.random.normal(0.0005, 0.018, n_days))),
    "BOND_ETF": 100 * np.exp(np.cumsum(np.random.normal(0.0002, 0.006, n_days))),
}, index=dates)

return_df = price_df.pct_change().dropna()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
price_df.plot(ax=axes[0], linewidth=1.2)
axes[0].set_title("Raw Prices")
axes[0].set_ylabel("Price")

return_df.plot(ax=axes[1], linewidth=0.9)
axes[1].set_title("Daily Returns")
axes[1].set_ylabel("Return")

plt.tight_layout()
plt.show()

## 2. Rebasing Prices

A common finance normalization is **rebasing**. This means setting all assets to the same starting value, often 100, and then tracking their relative growth.

### Formula

$$\text{Rebased Price}_t = 100 \times \frac{P_t}{P_0}$$

### Beginner explanation

Rebasing answers the question: **if I had invested the same starting amount in each asset, how would they compare over time?**

This is often the best normalization when you want to compare cumulative performance visually.

In [ ]:
rebased_prices = price_df.div(price_df.iloc[0]).mul(100)

rebased_prices.plot(title="Rebased Prices (Start = 100)", linewidth=1.3)
plt.ylabel("Rebased Level")
plt.tight_layout()
plt.show()

## 3. Min-Max Scaling

Min-max scaling rescales data to a fixed interval, often from 0 to 1.

### Formula

$$x_t^{\text{scaled}} = \frac{x_t - x_{\min}}{x_{\max} - x_{\min}}$$

### Beginner explanation

This method preserves the relative ordering of the data, but compresses everything into the same range. It is useful when you want to compare features with very different units.

### When to use it

Use min-max scaling when the model or visualization needs all variables on the same bounded scale. It is common in machine learning preprocessing and indicator dashboards.

In [ ]:
feature_df = pd.DataFrame({
    "Volatility": return_df.std() * np.sqrt(252),
    "Average Return": return_df.mean() * 252,
    "Price Level": price_df.iloc[-1],
})

minmax_df = (feature_df - feature_df.min()) / (feature_df.max() - feature_df.min())

display(feature_df.round(4))
display(minmax_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.heatmap(feature_df, annot=True, fmt=".2f", cmap="Blues", ax=axes[0])
axes[0].set_title("Raw Features")

sns.heatmap(minmax_df, annot=True, fmt=".2f", cmap="Greens", ax=axes[1])
axes[1].set_title("Min-Max Scaled Features")

plt.tight_layout()
plt.show()

## 4. Z-Score Standardisation

Z-score standardisation rescales data by centering it around the mean and dividing by the standard deviation.

### Formula

$$z_t = \frac{x_t - \mu}{\sigma}$$

### Beginner explanation

A z-score tells you how far an observation is from the average in units of standard deviation. A value of 2 means the observation is two standard deviations above the mean.

### When to use it

Use z-score standardisation when you want to compare values relative to their own history. In quant finance, this is common for signal generation, anomaly detection, spread trading, and factor comparison.

In [ ]:
zscore_returns = (return_df - return_df.mean()) / return_df.std(ddof=1)

display(zscore_returns.describe().round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
return_df.plot(ax=axes[0], linewidth=0.9)
axes[0].set_title("Raw Returns")
axes[0].set_ylabel("Return")

zscore_returns.plot(ax=axes[1], linewidth=0.9)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].axhline(2, color="red", linestyle="--", linewidth=1.0, alpha=0.7)
axes[1].axhline(-2, color="red", linestyle="--", linewidth=1.0, alpha=0.7)
axes[1].set_title("Z-Score Standardised Returns")
axes[1].set_ylabel("Z-Score")

plt.tight_layout()
plt.show()

## 5. When to Use Each Normalisation Method

| Method | Best Use | Main Question It Answers |
|---|---|---|
| **Rebasing** | Comparing cumulative performance | If all assets started at the same value, which performed better? |
| **Min-Max Scaling** | Machine learning and dashboards | How can I place features with different units on the same bounded scale? |
| **Z-Score Standardisation** | Signals and anomaly detection | How unusual is this observation relative to its own history? |

### Beginner explanation

There is no single “best” normalisation. The right one depends on your question. If you want to compare growth paths, rebase prices. If you want comparable features, use scaling. If you want to measure unusual behavior, use z-scores.

## 6. Final Takeaways

- Normalisation makes financial data easier to compare
- Raw prices are often misleading because assets live on different scales
- Rebasing is best for comparing performance paths
- Min-max scaling is useful for putting different features on the same range
- Z-scores are useful for measuring how unusual a value is relative to its history

### Final beginner takeaway

Normalisation does not change the economic meaning of the data. It changes the **scale** so that patterns become easier to compare, model, and interpret.